# Start by reading in a physics state:

In [ ]:
import xarray as xr
import ndsl.dsl.gt4py_utils as gt_utils
from ndsl import GridSizer, Quantity, QuantityFactory, TileCommunicator, TilePartitioner, NullComm, SubtileGridSizer
import ndsl.constants as constants

from pyshield.radiation.state import RTE_RRTMGPState
from pyshield._config import PHYSICS_PACKAGES
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
ds = xr.open_dataset("RESTART/restart_physics_state_0.nc")
ds2 = xr.open_dataset("RESTART/restart_dycore_state_0.nc")

In [ ]:
schemes = PHYSICS_PACKAGES

In [ ]:
nx_tile=20
ny_tile=20
nz=79
n_halo=3

In [ ]:
rank = 0

comm = NullComm(rank, 1)
communicator = TileCommunicator.from_layout(comm=comm, layout=(1,1))

sizer = SubtileGridSizer.from_tile_params(
    nx_tile=nx_tile,
    ny_tile=ny_tile,
    nz=nz,
    n_halo=n_halo,
    extra_dim_lengths={},
    layout=(1,1),
    tile_partitioner=communicator.partitioner.tile,
    tile_rank=communicator.tile.rank,
)
quantity_factory = QuantityFactory.from_backend(
    sizer, backend="numpy"
)

In [ ]:
state = RTE_RRTMGPState.init_zeros(quantity_factory, np)

In [ ]:
pkz = ds2.pkz.values
pe = ds2.pe.values
pk = ds2.pk.values

In [ ]:
p_lay = pkz[:,:,::-1]** (1./constants.KAPPA)

# Now we need certain things from the physics state:
 - level pressure in Pa
 - layer pressure in Pa
 - level temperature in K
 - layer temperature in K
 - surface temperature in K
 - water vapor mixing ratio in kg/kg
 - ozone mixing ratio in kg/kg

In [ ]:
print(ds.variables)

In [ ]:
ds.delp.shape

In [ ]:
prsi = ds.prsi.values[3:-4,3:-4,::-1] # level pressure
pt = ds.pt.values[3:-4,3:-4,-2::-1] # Layer temperature
delp = ds.delp.values[3:-4,3:-4,-2::-1]
delz = -1.* ds.delz.values[3:-4,3:-4,-2::-1]
dz = -1.* ds.dz.values[3:-4,3:-4,-2::-1]
qvap = ds.qvapor.values[3:-4,3:-4,-2::-1]
qliq = ds.qliquid.values[3:-4,3:-4,-2::-1]
qice = ds.qice.values[3:-4,3:-4,-2::-1]
qcld = ds.qcld.values[3:-4,3:-4,-2::-1]
phii = ds.phii.values[3:-4,3:-4,::-1]

Derive layer mean pressure from interface pressure

In [ ]:
prsl = (prsi[:,:,1:] - prsi[:,:,:-1]) / np.log(prsi[:,:,1:] / prsi[:,:,:-1])
prsl

Or from delp delz T and Q

In [ ]:
# prsl2 = delp / (prsi[:,:,:-1] - prsi[:,:,1:]) * constants.RDGAS * pt * (1 + constants.ZVIR * qvap)
# prsl2

Get the level (interface) temperatures too

In [ ]:
ptlev = np.zeros_like(prsi)
ptlev[:,:,1:-1] = pt[:,:,:-1] + (pt[:,:,1:] - pt[:,:,:-1]) * (np.log(prsi[:,:,1:-1]) - np.log(prsl[:,:,:-1])) / (np.log(prsl[:,:,1:]) - np.log(prsl[:,:,:-1]))
ptlev[:,:,-1] = pt[:,:,-1]
ptlev[:,:,0] = pt[:,:,0]

In [ ]:
qo3mr = ds.qo3mr.values
tskin = ptlev[:,:,-1]

And populate the state

In [ ]:
type(state.prsi)

In [ ]:
xds = state.to_rterrtmgp_xr()

In [ ]:
mu = np.sin(np.arange(400) * np.pi / 400.) - 0.3

In [ ]:
# xds.mu0.data[:] = 0.86
xds.mu0.data[:] = mu
xds.albedo.data[:] = 0.06
xds.sfc_emis.data[:] = 0.98

In [ ]:
xds.prsi.data = prsi.reshape(-1, 80)
xds.prsl.data = prsl.reshape(-1, 79)
xds.tlyr.data = pt.reshape(-1, 79)
xds.tlvl.data = ptlev.reshape(-1, 80)
xds.tsfc.data = tskin.reshape(-1)
xds.qvapor.data = qvap.reshape(-1, 79)
xds.qliquid.data = qliq.reshape(-1, 79)
xds.qice.data = qice.reshape(-1, 79)
# xds.qsnow.data = qice
# xds.qrain.data = qice
# xds.qgraupel.data = qice
xds

We need some other gases:

In [ ]:
gas_values = {
    "co2": 348e-6,
    "ch4": 1650e-9,
    "n2o": 306e-9,
    "n2": 0.7808,
    "o2": 0.2095,
    "co": 0.0,
}
for gas_name, value in gas_values.items():
    xds[gas_name] = value
xds

# Time to get the radiation going

In [ ]:
# from pyrte_rrtmgp import rrtmgp_cloud_optics, rrtmgp_gas_optics
# from pyrte_rrtmgp.data_types import (
#     CloudOpticsFiles,
#     GasOpticsFiles,
#     OpticsProblemTypes,
# )
# from pyrte_rrtmgp.rte_solver import rte_solve
# from pyrte_rrtmgp.examples import (
#     compute_RCE_clouds,
#     compute_RCE_profiles,
#     ALLSKY_EXAMPLES,
#     load_example_file,
# )
# from pyrte_rrtmgp.data_validation import AtmosphericMapping
# from pyrte_rrtmgp.config import (
#     DEFAULT_DIM_MAPPING,
#     DEFAULT_VAR_MAPPING,
# )
from pyrte_rrtmgp.rrtmgp import GasOptics, CloudOptics
from pyrte_rrtmgp.config import DEFAULT_DIM_MAPPING
from pyrte_rrtmgp.rrtmgp_data_files import CloudOpticsFiles, GasOpticsFiles
from pyrte_rrtmgp import rte
from pyrte_rrtmgp.input_mapping import AtmosphericMapping

In [ ]:
cloud_optics_lw = CloudOptics(
    cloud_optics_file=CloudOpticsFiles.LW_BND
)
gas_optics_lw = GasOptics(
    gas_optics_file=GasOpticsFiles.LW_G256
)

cloud_optics_sw = CloudOptics(
    cloud_optics_file=CloudOpticsFiles.SW_BND
)
gas_optics_sw = GasOptics(
    gas_optics_file=GasOpticsFiles.SW_G224
)

In [ ]:
gas_mapping = {
    "h2o": "qvapor",
    "o3": "qo3mr",
    'co':'co',
    'n2o': 'n2o',
    'o2': 'o2',
    'co2': 'co2',
    'n2': 'n2'
}
var_mapping = {
    "pres_layer": "prsl",
    "pres_level": "prsi",
    "temp_layer": "tlyr",
    "temp_level": "tlvl",
    "surface_temperature": "tsfc",
    "solar_zenith_angle": "solar_zenith_angle",
    "surface_albedo": "surface_albedo",
    "surface_albedo_direct": "surface_albedo_direct",
    "surface_albedo_diffuse": "surface_albedo_diffuse",
    "surface_emissivity": "surface_emissivity",
    "surface_emissivity_jacobian": "surface_emissivity_jacobian",   
}
atm_map = AtmosphericMapping(dim_mapping=DEFAULT_DIM_MAPPING, var_mapping=var_mapping,)

## And compute some optics:

In [ ]:
optical_props = gas_optics_lw.compute(
    xds, 
    problem_type=rte.OpticsTypes.ABSORPTION,
    add_to_input=False,
    gas_name_map=gas_mapping,
    variable_mapping=atm_map,
)



In [ ]:
optical_props

In [ ]:
optical_props["surface_emissivity"] = xds["sfc_emis"]

In [ ]:
clr_fluxes_lw = optical_props.rte.solve(add_to_input=False)
clr_fluxes_lw

In [ ]:
plt.plot(clr_fluxes_lw.lw_flux_up.isel(column=0),   clr_fluxes_lw.level, label="Flux up")
plt.plot(clr_fluxes_lw.lw_flux_down.isel(column=0), clr_fluxes_lw.level, label="Flux down")
plt.legend(frameon=False)

and the SW fluxes:

In [ ]:
sw_optics = gas_optics_sw.compute(
    xds, 
    problem_type=rte.OpticsTypes.TWO_STREAM,
    add_to_input=False,
    gas_name_map=gas_mapping,
    variable_mapping=atm_map,
)
sw_optics

In [ ]:
sw_optics["surface_albedo"] = xds["albedo"]
sw_optics["mu0"] = xds["mu0"]
fluxes_sw = sw_optics.rte.solve(add_to_input=False)

In [ ]:
plt.plot(fluxes_sw.sw_flux_up.isel(column=0), fluxes_sw.level, label="night SW clear flux up")
plt.plot(fluxes_sw.sw_flux_down.isel(column=0), fluxes_sw.level, label="night SW clear flux down")
plt.plot(fluxes_sw.sw_flux_up.isel(column=200), fluxes_sw.level, label="noon SW clear flux up")
plt.plot(fluxes_sw.sw_flux_down.isel(column=200), fluxes_sw.level, label="noon SW clear flux down")
plt.legend(frameon=False)